In [ ]:
import os
import zipfile

# google.colab only exists in Colab — not when running this notebook in PyCharm/Jupyter
try:
    from google.colab import drive

    drive.mount("/content/drive")
    zip_path = "/content/drive/MyDrive/R_D-CV/Drones_Datasets.zip"
    with zipfile.ZipFile(zip_path) as z:
        z.extractall("/content")
    os.chdir("/content")
except ImportError:
    # Local run: dataset should already be next to this notebook
    notebook_dir = os.path.dirname(os.path.abspath("train_model_drones_dataset.ipynb"))
    os.chdir(notebook_dir)
    assert os.path.exists("Drones_Datasets/data.yaml"), (
        "Drones_Datasets/data.yaml not found. Open the notebook from model/ "
        "or put Drones_Datasets next to it."
    )
    print("Not in Colab — using local Drones_Datasets/")

In [ ]:
import torch
from ultralytics import YOLO

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — Runtime → Change runtime type → T4 GPU, then re-run all cells")

In [ ]:
# Faster Colab settings: small model, fewer epochs, slightly smaller images
model = YOLO("yolo11n.pt")

model.train(
    data="Drones_Datasets/data.yaml",
    batch=16,       # raise if GPU has headroom; lower to 8 if OOM
    epochs=30,      # 50 is slow; raise later if needed
    imgsz=416,      # 640 is slower; use 640 for final quality
    name="drones_datasets_yolov11",
    save=True,
    patience=10,
    save_period=-1,
    val=True,
    workers=2,
    cache=False,
    amp=True,       # mixed precision — faster on GPU
    device=0 if torch.cuda.is_available() else "cpu",
)

In [ ]:
import matplotlib.pyplot as plt
import os
import random
import cv2

model = YOLO("runs/detect/drones_datasets_yolov11/weights/best.pt")

images_path = "Drones_Datasets/train/images"

fig, ax = plt.subplots(4, 2, figsize=(15, 16))
ax = ax.ravel()

img_files = [
    f for f in os.listdir(images_path)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
]

for i in range(8):
    img_file_name = random.choice(img_files)
    img_path = os.path.join(images_path, img_file_name)

    img = cv2.imread(img_path)
    if img is None:
        continue

    results = model(img)[0]

    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    ax[i].imshow(img)
    ax[i].axis("off")

plt.show()